In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import VotingClassifier

In [2]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

X = iris.data   # 입력 특성
y = iris.target # 정답: 0=setosa, 1=versicolor, 2=virginica

In [3]:
clf1 = LogisticRegression(random_state=1)
clf2 = DecisionTreeClassifier(random_state=1, max_depth=4)

eclf = VotingClassifier(estimators=[('lr', clf1), ('dt', clf2)], voting='hard')

In [4]:
from sklearn.model_selection import cross_val_score

ev1 = cross_val_score(clf1, X, y, cv=5).mean()
ev2 = cross_val_score(clf2, X, y, cv=5).mean()

ev3 = cross_val_score(eclf, X, y, cv=5)

print(ev1)
print(ev2)
print("-"*60)
print(ev3)
print(ev3.mean())

0.9733333333333334
0.9666666666666668
------------------------------------------------------------
[0.96666667 0.96666667 0.9        1.         1.        ]
0.9666666666666668


C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
 

In [5]:
c_params = [0.1, 5.0, 7.0, 10.0, 15.0, 20.0, 100.0]

params ={
    "lr__solver": ["liblinear"],          # 로지스틱 회귀: 최적화 알고리즘
    "lr__penalty": ["l2"],                # 로지스틱 회귀: 규제 방식(L2 규제)
    "lr__C": c_params,                   # 로지스틱 회귀: 규제 강도의 역수(작을수록 강한 규제)
    
    "dt__criterion": ["gini", "entropy"], # 의사결정트리: 노드 분할의 불순도 측정 기준
    "dt__max_depth": [10, 8, 7, 6, 5, 4, 3, 2],  # 의사결정트리: 트리의 최대 깊이
    "dt__min_samples_leaf": [1, 2, 3, 4, 5, 6, 7, 8, 9]  # 의사결정트리: 리프 노드의 최소 샘플 수
    }

clf1 = LogisticRegression(random_state=1)
clf2 = DecisionTreeClassifier(random_state=1)
eclf = VotingClassifier(estimators=[('lr', clf1), ('dt', clf2)], voting='hard')

In [6]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(estimator=eclf, param_grid=params, cv=5)
grid = grid.fit(X, y)

print(grid.best_score_)
print(grid.best_params_)

0.9666666666666668
{'dt__criterion': 'gini', 'dt__max_depth': 10, 'dt__min_samples_leaf': 1, 'lr__C': 0.1, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}


In [7]:
from sklearn.metrics import classification_report

# GridSearchCV 모델에 베스트 모델이 저장되어있기 때문에 아래 코드는 실행 안해도 됨
#clf1 = LogisticRegression(C= '0.1',penalty='l2', lr__solver="liblinear", random_state=1)
#clf2 = DecisionTreeClassifier(criterion = 'gini', max_depth='10', min_samples_leaf = '1', random_state=1)
#eclf = VotingClassifier(estimators=[('lr', clf1), ('dt', clf2)], voting='hard')
#eclf.fit(X, y)
#pred = eclf.predict(X)

pred = grid.best_estimator_.predict(X)
print(classification_report(y, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        50
           2       1.00      1.00      1.00        50

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



In [8]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(n_jobs=7, oob_score=True)

params ={
    "n_estimators" : [10, 20, 30, 50, 100],
    "max_features" : np.arange(1,X.shape[1] + 1)
    }

In [9]:
grid = GridSearchCV(estimator=rfc, param_grid=params, cv=5)
grid = grid.fit(X, y)

print(grid.best_score_)
print(grid.best_params_)
print(grid.best_estimator_.oob_score_)

C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\ensemble\_forest.py:615: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\ensemble\_forest.py:615: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\ensemble\_forest.py:615: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\ensemble\_forest.py:615: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
C:\Users\DANA\.conda\envs\ml\lib\site-packages\sklearn\ensemble\_forest.py:615: UserWarning: Some inputs do not have OOB

0.9666666666666668
{'max_features': 2, 'n_estimators': 50}
0.9533333333333334


In [10]:
# GridSearchCV 모델에 베스트 모델이 저장되어있기 때문에 아래 코드는 실행 안해도 됨
#rf  = RandomForestClassifier(n_jobs=7, oob_score=True, max_features= 1, n_estimators = 20)
#rf.fit(X,y)
#pred = rf.predict(X)

pred = grid.best_estimator_.predict(X)
print(classification_report(y, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        50
           2       1.00      1.00      1.00        50

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



In [11]:
import pandas as pd

wine = pd.read_csv("https://raw.githubusercontent.com/prof-dana/Lecture-Code/refs/heads/main/AdvancedMachineLearning/data/wine.csv")
wine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  6497 non-null   object 
 1   fixed acidity         6497 non-null   float64
 2   volatile acidity      6497 non-null   float64
 3   citric acid           6497 non-null   float64
 4   residual sugar        6497 non-null   float64
 5   chlorides             6497 non-null   float64
 6   free sulfur dioxide   6497 non-null   float64
 7   total sulfur dioxide  6497 non-null   float64
 8   density               6497 non-null   float64
 9   pH                    6497 non-null   float64
 10  sulphates             6497 non-null   float64
 11  alcohol               6497 non-null   float64
 12  quality               6497 non-null   int64  
dtypes: float64(11), int64(1), object(1)
memory usage: 660.0+ KB


In [12]:
wine['type'] = wine['type'].map({'red': 0, 'white': 1})
wine.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,0,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,0,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,0,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#예측값 분리
Y = wine['type']
X = wine.iloc[:, 1:]

#훈련 및 테스트 데이터 분리
trX, teX, trY, teY = train_test_split(
       X, Y, test_size=0.2, random_state=42)

#정규화
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(trX)
tr_scaled = ss.transform(trX)
te_scaled = ss.transform(teX)
tr_scaled[:5]

array([[-0.46822898, -0.59951186,  0.20546655,  0.4646116 , -0.71280142,
         0.31165933,  0.34356569, -0.29614553, -0.16888369, -1.0832659 ,
         0.01376574, -0.92920628],
       [ 0.85149992, -0.35284877,  1.10406349, -0.70299854,  1.06747   ,
        -1.4087346 , -1.84751841, -0.20307163,  0.26967864,  0.59857677,
         1.60871671,  1.35243087],
       [ 0.38571325,  2.32961232, -2.14471006, -0.70299854,  0.2347624 ,
         0.02492701, -1.29529396, -0.32606213,  1.20945505,  0.26220824,
         1.10504798,  0.2116123 ],
       [-1.55506455,  0.20214318,  0.06722087, -0.89065017, -0.79894359,
        -1.00730935, -0.61837367, -1.49280991,  0.958848  , -1.01599219,
         1.0211032 ,  0.2116123 ],
       [-0.46822898, -0.47618032,  1.65704622,  2.07007555, -0.0810922 ,
         0.08227347,  0.45044784,  1.1332036 , -0.67009777, -0.27598142,
        -0.99357172, -0.92920628]])

In [14]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, tr_scaled, trY, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9993746530912755 0.9932651588065449


In [15]:
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gb, tr_scaled, trY, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9996632649507319 0.994997778929444


In [16]:
gb.fit(tr_scaled, trY)#학습
prY = gb.predict(te_scaled)#예측
print(classification_report(teY, prY)) #평가

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       341
           1       0.99      0.99      0.99       959

    accuracy                           0.99      1300
   macro avg       0.98      0.99      0.99      1300
weighted avg       0.99      0.99      0.99      1300

